# NoUs on Colab — 手動・個別実行用

普段は使いません。**通常は `nous_colab.ipynb`（フォームで選ぶだけ）を使ってください。** ここは、データを初めて作る・作り直すときの個別の手順、討論、人による採点、知識グラフのダウンロードなど、フォームにない操作用です。上から順に実行します（各ステップのあとに `save()` が入っています）。

---
## ★ 5. 初回・再構築用（普段は不要）
以下は、データを初めて作るとき・作り直すときの手順です。`準備 1〜7` は、ワンセットを使うなら不要です。

## 準備 1. GPU の確認（任意）
`メニュー → ランタイム → ランタイムのタイプを変更` で GPU を選ぶと LLM が速くなります（T4 は無料枠。L4 / A100 / G4 は有料でユニットを消費）。

In [ ]:
!nvidia-smi || echo 'GPU なし（CPU でも動きますが遅いです）'

## 準備 2. Google Drive を接続
許可画面が出たら、自分のアカウントを選んで許可してください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 準備 3. 秘密情報（APIキー）の読み込み
左の 🔑（シークレット）に以下を登録し、「ノートブックからのアクセス」をオンにしてください。

| 名前 | 値 | 必須 |
|---|---|---|
| `OPENALEX_API_KEY` | OpenAlex の API キー | 推奨 |
| `GH_TOKEN` | GitHub の Personal Access Token | リポジトリが非公開の場合のみ |

In [ ]:
import os
from google.colab import userdata

def _secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

for name in ("OPENALEX_API_KEY", "GH_TOKEN"):
    value = _secret(name)
    if value:
        os.environ[name] = value
    print(f"{name:18s}: {'OK' if value else '未設定'}")

## 準備 4. コードを取得

In [ ]:
import os
BRANCH = "main"
token = os.environ.get("GH_TOKEN")
import base64, subprocess
def git(*args, token=None, cwd=None, quiet=False):
    """git を実行する。トークンは URL にも引数にも入れず（http.extraheader を環境変数で渡す）、出力からは伏せ字にする。"""
    env = dict(os.environ)
    secret = []
    if token:
        b = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        secret = [token, b]
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {b}")
    r = subprocess.run(["git", *args], capture_output=True, text=True, env=env, cwd=cwd)
    out = (r.stdout + r.stderr).strip()
    for s in secret:
        out = out.replace(s, "***")
    if out and not (quiet and r.returncode):
        print(out)
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed (exit {r.returncode})")
def git_auto(*args, token=None, cwd=None):
    """公開リポジトリは認証なしで足りる（無効なトークンを送ると 401 で失敗する）。失敗したときだけトークンで再試行する。"""
    try:
        git(*args, cwd=cwd, quiet=bool(token))
    except RuntimeError:
        if not token:
            raise
        git(*args, token=token, cwd=cwd)

if os.path.exists("/content/NoUs/.git"):
    git_auto("fetch", "-q", "origin", token=token, cwd="/content/NoUs")
    git("checkout", "-q", BRANCH, cwd="/content/NoUs")
    git_auto("pull", "-q", "origin", BRANCH, token=token, cwd="/content/NoUs")
else:
    git_auto("clone", "-q", "-b", BRANCH, "https://github.com/kohga1220-stack/NoUs.git", "/content/NoUs", token=token)
os.chdir("/content/NoUs")
git("log", "--oneline", "-1")

## 準備 5. ライブラリをインストール（2〜4分）

In [ ]:
!pip install -q -r requirements.txt pytest
!python -m pytest -q 2>&1 | tail -1

## 準備 6. Ollama（ローカル LLM）を起動（初回 3〜5分）

In [ ]:
import subprocess, time, shutil
if not shutil.which("ollama"):
    !apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull gemma4:e2b
!ollama list

## 準備 7. データを Drive から読み込む ＋ `save()` を定義
- 初回：旧データ `マイドライブ/AI/projects/nous/data` をコピー（元データはそのまま残ります）
- 2回目以降：`マイドライブ/AI/projects/NoUs_data`（`save()` の保存先）から読み込み

In [ ]:
import os
from nous.colab_setup import prepare_data, make_save
OLD_DATA   = "/content/drive/MyDrive/AI/projects/nous/data"
DRIVE_DATA = "/content/drive/MyDrive/AI/projects/NoUs_data"
LOCAL_DATA = "/content/NoUs/data"
print(prepare_data(LOCAL_DATA, DRIVE_DATA, OLD_DATA))
save = make_save(LOCAL_DATA, DRIVE_DATA)
print(os.listdir(LOCAL_DATA))


### 実行（各ステップのあとに `save()` が入っています）

## A. 知識ベースの確認（旧データの索引を更新）

In [ ]:
!python nous.py embed
save()

## B. 構造の抽象化（まず10件で試す）
`[physics] Phase transition → threshold, ...` のように出れば成功。

In [ ]:
!python nous.py abstract 10

問題なければ残りすべて（件数により数十分）

In [ ]:
!python nous.py abstract
save()

## C. OpenAlex から各分野の論文を収集（まず各分野5件）

In [ ]:
!python nous.py openalex 5
!python nous.py embed
!python nous.py abstract
save()

## D. 分野間の空白地帯（Void）
**この出力をチャットに貼ってください。**

In [ ]:
!python nous.py voids --refresh
save()

## E. 空白地帯を架橋する論文の収集 ＋ 分野トレンド

In [ ]:
!python nous.py fillvoids --openalex
!python nous.py embed
!python nous.py abstract
!python nous.py trends
save()

## F. 討論（1回数分）
テーマは D の結果から選ぶのがおすすめ。`TOPIC` を書き換えて実行。

In [ ]:
TOPIC = "phase transition"
!python nous.py debate "{TOPIC}"
save()

## G. LLM 審査員による評価

In [ ]:
!python nous.py evaluate
!python nous.py reliability
save()

## G2. 分野名の統一（1回だけ実行）
旧データの `physics` などを OpenAlex の分野名にそろえます。何度実行しても安全です。

In [ ]:
!python nous.py migrate-domains
save()

## G3. 文献チェック（OpenAlex）
各仮説に最も近い既存論文を探し、文献に照らした新規性（lit-novelty）を記録します。1件約1秒・$0.001。

In [ ]:
!python nous.py litcheck
save()

## G3b. lit-novelty の較正（1回）
有名な理論10個を同じ方法で測り、「既出」の目安を作ります。約$0.01。`scores` に ⚑ が付くようになります。

In [ ]:
!python nous.py calibrate
save()

## G3c. 組み合わせのチェック
仮説を2〜4個の概念に分け、概念ペアごとに「両方を扱った論文が何本あるか」と「2つの概念の論文群が近い分野にあるか（隣接性）」を OpenAlex で調べます。隣接性が高いのに共起が少ない／ない組み合わせに ★（橋渡しの候補）が付きます。API費用は数セント。
最初に、有名な組み合わせと無関係な組み合わせの実測値が目盛りとして表示されます。すでに調べた仮説は再取得せず、保存済みの件数から判定し直します。

In [ ]:
!python nous.py combocheck
save()

## G4. 別モデルの審査員を追加（任意）
仮説を作ったモデルとは別のモデルで採点すると、より独立した評価になります。

In [ ]:
!ollama pull llama3.2:3b
!ollama pull qwen2.5:3b
!python nous.py evaluate --judges gemma4:e2b,llama3.2:3b,qwen2.5:3b
save()

## H. 自分で採点（10件程度）
入力欄が出たら 1〜5 を入力して Enter。`s` = この仮説を飛ばす、`q` = 終了。

In [ ]:
from nous.evaluation.runner import human_rate
human_rate("kohga", limit=10)
save()

## I. 結果の確認
`scores` と `reliability` の出力もチャットに貼ってください。

In [ ]:
!python nous.py scores
!python nous.py reliability

## （任意）知識グラフの HTML をダウンロード

In [ ]:
!python nous.py graph
from google.colab import files
files.download("data/nous_full_graph.html")